# Classer un sentiment avec TF-IDF

**Objectifs.** Décrire une tâche texte, comparer un vocabulaire aux tokens, ajuster TF-IDF dans une pipeline train-only et inspecter les erreurs. Durée : 75 min. Le micro-corpus est écrit pour illustrer le protocole ; il ne représente pas une population ni un usage client réel.

## 1. Définir le problème

Chaque ligne est un texte court et un label binaire positif/négatif. Le modèle ne comprend pas l’ironie ni le contexte ; nous voulons d’abord tester si le pipeline fonctionne. Les doublons et formulations artificielles limitent la portée du résultat.

In [ ]:
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

texts = [
 'excellent service and kind staff', 'the meal was fresh and delicious', 'fast delivery and great quality', 'friendly team solved my issue',
 'wonderful experience and helpful support', 'the product works perfectly', 'clear instructions and easy setup', 'I am happy with this purchase',
 'terrible service and rude staff', 'the meal was cold and tasteless', 'slow delivery and poor quality', 'unhelpful team ignored my issue',
 'awful experience and frustrating support', 'the product broke immediately', 'confusing instructions and difficult setup', 'I regret this purchase',
 'great quality but the box was damaged', 'helpful support but delivery was slow', 'fresh food and friendly staff', 'poor product and confusing manual',
 'fast shipping and perfect item', 'rude response and long wait', 'kind team and useful solution', 'cold food arrived late'
]
labels = [1,1,1,1,1,1,1,1,0,0,0,0,0,0,0,0,1,0,1,0,1,0,1,0]
frame = pd.DataFrame({'text': texts, 'sentiment': labels})
print(frame.groupby('sentiment').size())
frame.head()

## 2. Split avant l’apprentissage du vocabulaire

`TfidfVectorizer` découvre son vocabulaire et ses fréquences documentaires pendant `fit`. Il doit être dans la pipeline pour que les mots du test ne soient pas appris à l’avance.

In [ ]:
text_train, text_test, y_train, y_test = train_test_split(
    frame['text'], frame['sentiment'], test_size=0.25, stratify=frame['sentiment'], random_state=12
)
text_baseline = DummyClassifier(strategy='most_frequent').fit(text_train.to_frame(), y_train)
text_model = make_pipeline(
    TfidfVectorizer(ngram_range=(1, 2), min_df=1),
    LogisticRegression(C=1.0, max_iter=1_000)
)
text_model.fit(text_train, y_train)
prediction = text_model.predict(text_test)
print('majority baseline accuracy:', text_baseline.score(text_test.to_frame(), y_test))
print(classification_report(y_test, prediction, zero_division=0))
print('confusion matrix:', confusion_matrix(y_test, prediction).tolist())

## 3. Inspecter les coefficients et erreurs

Le coefficient mesure une association dans ce corpus et ce modèle régularisé ; ce n’est pas une explication causale. Des n-grams familiers peuvent masquer que le test est petit.

In [ ]:
vectorizer = text_model.named_steps['tfidfvectorizer']
classifier = text_model.named_steps['logisticregression']
weights = pd.Series(classifier.coef_[0], index=vectorizer.get_feature_names_out())
print('termes positifs associés :\n', weights.nlargest(8).round(2))
print('termes négatifs associés :\n', weights.nsmallest(8).round(2))
error_review = pd.DataFrame({'text': text_test, 'actual': y_test, 'predicted': prediction})
error_review[error_review['actual'] != error_review['predicted']]

## 4. Expérience et limites

Comparez unigrammes à unigrammes+bigrammes en validation croisée, en plaçant chaque `TfidfVectorizer` dans une pipeline. Ne choisissez pas à partir du test. Tokenization avancée, embeddings, LSTM, self-attention et Transformers/Hugging Face sont les suites du module et nécessitent une leçon distincte.